# Phase 8: Final Evaluation on the Test Data
The saved model bundle with the thresholds chosen on the validation data is applied
to the official test split exactly once. No changes are made afterwards.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.metrics import (
    average_precision_score, f1_score, precision_score, recall_score, roc_auc_score,
)

from portfolio_projekt.data import THRESHOLD, load_english
from portfolio_projekt.recommend import load_bundle, recommend

In [ ]:
_, _, df_test = load_english()
bundle = load_bundle()

X_test = bundle["vectorizer"].transform(df_test["text"])
probs = pd.DataFrame({label: model.predict_proba(X_test)[:, 1] for label, model in bundle["models"].items()})
print(probs.shape)

In [ ]:
val_pr_auc = {"toxicity": 0.677, "obscene": 0.506, "threat": 0.175,
              "insult": 0.675, "identity_attack": 0.295, "sexual_explicit": 0.214}

rows = []
for label in bundle["labels"]:
    y_true = (df_test[label] >= THRESHOLD).values
    p = probs[label].values
    y_pred = p >= bundle["thresholds"][label]

    rows.append({
        "label": label,
        "pr_auc_val": val_pr_auc[label],
        "pr_auc_test": average_precision_score(y_true, p),
        "roc_auc_test": roc_auc_score(y_true, p),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "positives": int(y_true.sum()),
    })

df_final = pd.DataFrame(rows)
df_final.round(3)

In [ ]:
levels = [recommend(row, bundle["thresholds"])["level"] for row in probs.to_dict("records")]

df_levels = pd.DataFrame({
    "level": levels,
    "truly_toxic": (df_test["toxicity"] >= THRESHOLD).values,
})
df_levels.groupby("level").agg(
    comments=("truly_toxic", "size"),
    share_truly_toxic=("truly_toxic", "mean"),
).round(3)

**Final results (test data, used once):** Test PR-AUC matches validation closely
(toxicity 0.673 vs. 0.677, threat 0.164 vs. 0.175), so the estimates are reliable.
At the fixed thresholds, toxicity reaches precision 0.60 and recall 0.65; threat only
0.21 and 0.26.
Recommendations: "Delete" is justified in 90 % of cases. Level 3 (37 %) is less precise
than level 2 (53 %) because identity-attack signals are biased; level 5 is correct in
38 % of cases (weak threat model), which is why a human decides there.
Levels 2–5 cover ~9 % of all comments and contain ~65 % of all toxic comments.